In [ ]:
from ac_segmentation.methods.segment_array import segment_gunpowder
from ac_segmentation.methods.equalize_array import adjust_contrast_gunpowder
from ac_segmentation.methods.fuse_volume import fuse_gunpowder
from ac_segmentation.utils.tensorstore import *
import json
import numpy as np

In [ ]:
###Create input tensor
arr = np.random.rand(100,100,100).astype('uint8')
input_tensor = create_tensor(fpath="input_vol/0", 
                             arr_shape=(100,100,100), 
                             dtype='uint8', 
                             chunk_shape=[64, 64, 64], 
                             driver='zarr3', 
                             codecs={"name": "blosc", "configuration": {"cname": "lz4", "clevel": 4}}, 
                             sharded=True, 
                             shard_factor=16)
input_tensor[...].write(arr).result()

In [ ]:
###Create output tensor
output_tensor = create_tensor(fpath="output_vol/0", 
                             arr_shape=(100,100,100), 
                             dtype='uint8', 
                             chunk_shape=[64, 64, 64], 
                             driver='zarr3', 
                             codecs={"name": "blosc", "configuration": {"cname": "lz4", "clevel": 4}}, 
                             sharded=True, 
                             shard_factor=16)

In [ ]:
###Run segmentation 
segment_gunpowder(
            input_tensor,
            output_tensor,
            "model_files/segmentation/best.ckpt",
            iter_size=(64,64,64),
            batch_size=10,
            cutout=None,
            gpu_device=None,
            cpus=10,
            preprocess={'method': 'percentile', 'values': [96,97]},
            mask_file=None,
            add_margin=16)

In [ ]:
###Run contrast equalization
adjust_contrast_gunpowder(input_tensor, 
                          output_tensor, 
                          iter_size=(64,64,64), 
                          batch_size=10, 
                          cutout=None, 
                          preprocess={'method':'percentile','values':[5,99.5]}, 
                          mask_file=None, 
                          add_margin=32, depth=.9)  

In [ ]:
###Run fusion

#create input files and translations lists
translations = [[0,0,0],[10,10,10]]

file = os.path.join(os.getcwd(),"input_vol")
fpaths = [file,file]
                            
#open input tensors
arrays = []
for fpath in fpaths:
    #Open input tensor  
    in_path = os.path.join(fpath, '0')                                                                                             
    arrays.append(open_tensor(in_path)) 

fuse_gunpowder(arrs=arrays, 
               translations=translations, 
               output_path='fusion/0', 
               batch_size=10, 
               iter_size = (64,64,64)) 
